In [ ]:
# ==========================================================
# Установка и загрузка библиотек
# ==========================================================
!pip install autogluon.tabular --quiet

import pandas as pd
from autogluon.tabular import TabularPredictor
import os

# ==========================================================
# 1. Загрузка данных
# ==========================================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "target"     # <-- поменять под свою задачу
ID_COL = "id"          # <-- колонка, которая будет в сабмите

# ==========================================================
# 2. Обучение AutoML
# ==========================================================
save_path = "autogluon_model"

predictor = TabularPredictor(
    label=TARGET,
    eval_metric="auto",        # AutoGluon сам подберёт лучшую
    path=save_path
)

predictor.fit(
    train,
    time_limit=3600,           # 1 час (ставь что хочешь)
    presets="best_quality",    # самый сильный режим (увеличивает время)
    ag_args_fit={"num_gpus": 0}
)

# ==========================================================
# 3. Получение информации о моделях
# ==========================================================

print("\n=== Leaderboard (все модели) ===")
leaderboard = predictor.leaderboard(silent=True)
print(leaderboard)

print("\n=== Лучшая модель ===")
print(predictor.get_model_best())

print("\n=== Сводка обучения ===")
predictor.fit_summary()

# Фичеважность (работает и для регрессии, и для классификации)
print("\n=== Feature importance ===")
print(predictor.feature_importance(train))

# ==========================================================
# 4. Предсказания на тесте
# ==========================================================
test_pred = predictor.predict(test)

# ==========================================================
# 5. Генерация сабмита
# ==========================================================
submit = pd.DataFrame()
submit[ID_COL] = test[ID_COL]
submit[TARGET] = test_pred

submit.to_csv("submission.csv", index=False)
print("\nФайл submission.csv успешно создан.")
